In [ ]:
from pathlib import Path
import json
from collections import defaultdict

import numpy as np
import pandas as pd
from scipy.stats import ks_2samp, energy_distance, wasserstein_distance, gaussian_kde
from scipy.spatial.distance import jensenshannon

ROOT = Path.cwd()
RESULTS_ROOT = ROOT / "number_extracted_shuffling" / "shuffling_human"
GT_FILE = ROOT / "ground_truth_values" / "shuffling_human" / "ground_truth_values.json"

print("ROOT:", ROOT)
print("RESULTS_ROOT exists:", RESULTS_ROOT.exists())
print("GT_FILE exists:", GT_FILE.exists())

In [ ]:
def lehmer_code(perm: np.ndarray) -> np.ndarray:
    """
    Lehmer code L_i = #{ j>i : perm[j] < perm[i] }.
    perm: shape (n,)
    returns: shape (n,) with last digit always 0.
    """
    perm = np.asarray(perm)
    n = perm.shape[0]
    L = np.zeros(n, dtype=int)
    for i in range(n):
        if i + 1 < n:
            L[i] = int(np.sum(perm[i + 1 :] < perm[i]))
        else:
            L[i] = 0
    return L


def lehmer_encode_batch(perms: np.ndarray) -> np.ndarray:
    """
    perms: (m, n)
    returns: (m, n) Lehmer codes
    """
    perms = np.asarray(perms)
    if perms.ndim != 2:
        raise ValueError("perms must be a 2D array of shape (m, n)")
    m, _ = perms.shape
    return np.vstack([lehmer_code(perms[i]) for i in range(m)])


def lehmer_normalize(L: np.ndarray) -> np.ndarray:
    """
    Normalize Lehmer digits to [0,1] per coordinate:
      Z_i = L_i / (n - i - 1)  for i < n-1
      Z_{n-1} = 0
    L: (m, n) or (n,)
    returns same shape as L (float)
    """
    L = np.asarray(L)
    if L.ndim == 1:
        n = L.shape[0]
        denom = np.maximum(np.arange(n - 1, -1, -1), 1)
        Z = L.astype(float) / denom
        Z[-1] = 0.0
        return Z

    if L.ndim == 2:
        m, n = L.shape
        denom = np.maximum(np.arange(n - 1, -1, -1), 1).astype(float)
        Z = L.astype(float) / denom[None, :]
        Z[:, -1] = 0.0
        return Z

    raise ValueError("L must be 1D or 2D")

In [ ]:
def _canonical_value(v):
    if isinstance(v, (int, float)):
        if not np.isfinite(v):
            return None
        # Normalize numeric types for stable keying
        return float(v)
    if isinstance(v, str):
        return v.strip()
    return None


def build_value_mapping(values):
    keys = []
    for v in values:
        key = _canonical_value(v)
        if key is None:
            return None
        keys.append(key)
    if len(keys) != len(set(keys)):
        return None
    return {k: i for i, k in enumerate(keys)}


def list_to_perm_indices(values, value_to_index):
    if not isinstance(values, list):
        return None
    indices = []
    for v in values:
        key = _canonical_value(v)
        if key is None or key not in value_to_index:
            return None
        indices.append(value_to_index[key])
    if len(indices) != len(value_to_index):
        return None
    if len(set(indices)) != len(indices):
        return None
    return np.asarray(indices, dtype=int)


def list_to_perm_indices_checked(values, value_to_index):
    if not isinstance(values, list):
        return None, "not_list"
    indices = []
    for v in values:
        key = _canonical_value(v)
        if key is None:
            return None, "bad_value"
        if key not in value_to_index:
            return None, "value_not_in_gt"
        indices.append(value_to_index[key])
    if len(indices) != len(value_to_index):
        return None, "length_mismatch"
    if len(set(indices)) != len(indices):
        return None, "duplicate_values"
    return np.asarray(indices, dtype=int), "ok"


def normalize_gt_perms(gt_values):
    if not isinstance(gt_values, list) or len(gt_values) == 0:
        return None
    perms = gt_values if isinstance(gt_values[0], list) else [gt_values]
    value_to_index = build_value_mapping(perms[0])
    if value_to_index is None:
        return None
    perm_indices = []
    for perm in perms:
        perm_idx = list_to_perm_indices(perm, value_to_index)
        if perm_idx is None:
            return None
        perm_indices.append(perm_idx)
    return value_to_index, perm_indices


def ks_score(a, b):
    """Calculate Kolmogorov-Smirnov test."""
    stat, p_value = ks_2samp(a, b)
    return {
        "ks_statistic": float(stat),
        "ks_p_value": float(p_value),
    }


def _w1_sorted_equal(x_sorted, y_sorted):
    """Wasserstein-1 between two equal-size 1D samples, both pre-sorted."""
    return np.mean(np.abs(x_sorted - y_sorted))


def _w1_sorted(x, y):
    """Wasserstein-1 for 1D samples of possibly unequal size. Sorts internally."""
    x = np.sort(x)
    y = np.sort(y)
    if x.size == y.size:
        return np.mean(np.abs(x - y))
    # Fallback: scipy handles unequal sizes via the CDF-integral form
    return wasserstein_distance(x, y)


def distance_distribution_scores(a, b, n_resamples=999, rng=20240521):
    """Debiased Wasserstein-1 and energy distance with a shared permutation null."""
    rng = np.random.default_rng(rng)
    a = np.asarray(a, dtype=float).ravel()
    b = np.asarray(b, dtype=float).ravel()

    n_a = a.size
    n_b = b.size
    if n_a < 2 or n_b < 2:
        return {k: float("nan") for k in (
            "wasserstein_debiased", "wasserstein_z",
            "energy_debiased", "energy_z",
        )}

    pooled = np.concatenate([a, b])
    n_total = pooled.size
    equal_sizes = (n_a == n_b)

    # Observed statistics
    if equal_sizes:
        w_obs = _w1_sorted_equal(np.sort(a), np.sort(b))
    else:
        w_obs = _w1_sorted(a, b)
    e_obs = float(energy_distance(a, b))

    # Shared permutation null
    w_null = np.empty(n_resamples)
    e_null = np.empty(n_resamples)
    for i in range(n_resamples):
        perm = rng.permutation(n_total)
        x = pooled[perm[:n_a]]
        y = pooled[perm[n_a:]]
        if equal_sizes:
            w_null[i] = _w1_sorted_equal(np.sort(x), np.sort(y))
        else:
            w_null[i] = _w1_sorted(x, y)
        e_null[i] = energy_distance(x, y)

    def _summarize(obs, null):
        mean = null.mean()
        std = null.std()
        debiased = float(obs - mean)
        if std < 1e-12:
            z = float("nan")
        else:
            z = float((obs - mean) / std)
        return debiased, z

    w_debiased, w_z = _summarize(w_obs, w_null)
    e_debiased, e_z = _summarize(e_obs, e_null)

    return {
        "wasserstein_debiased": w_debiased,
        "wasserstein_z": w_z,
        "energy_debiased": e_debiased,
        "energy_z": e_z,
    }


def js_divergence_score(a, b, grid_size=512, pad=0.1):
    """Jensen-Shannon divergence via KDE on a shared grid."""
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)

    if a.size < 2 or b.size < 2:
        return np.nan
    if np.allclose(a.min(), a.max()) and np.allclose(b.min(), b.max()):
        return 0.0 if np.isclose(a[0], b[0]) else np.nan

    lo = min(a.min(), b.min())
    hi = max(a.max(), b.max())
    span = hi - lo
    lo -= pad * span
    hi += pad * span
    grid = np.linspace(lo, hi, grid_size)

    try:
        p = gaussian_kde(a)(grid)
        q = gaussian_kde(b)(grid)
    except (np.linalg.LinAlgError, ValueError):
        return np.nan

    p_sum, q_sum = p.sum(), q.sum()
    if p_sum == 0 or q_sum == 0:
        return np.nan
    p /= p_sum
    q /= q_sum

    js_distance = jensenshannon(p, q)
    return float(js_distance ** 2)


def _lehmer_l0_distribution(perms):
    if perms is None or len(perms) == 0:
        return None
    arr = np.vstack(perms)
    L = lehmer_encode_batch(arr)
    return L[:, 0].astype(float)


def perm_distance_metrics(gt_perms, pred_perms):
    gt_l0 = _lehmer_l0_distribution(gt_perms)
    pred_l0 = _lehmer_l0_distribution(pred_perms)
    if gt_l0 is None or pred_l0 is None:
        return None

    ks = ks_score(gt_l0, pred_l0)
    dist_scores = distance_distribution_scores(gt_l0, pred_l0)

    rec = {
        "ks_statistic": ks["ks_statistic"],
        "ks_p_value": ks["ks_p_value"],
        "js_divergence": js_divergence_score(gt_l0, pred_l0),
    }
    rec.update(dist_scores)
    return rec

In [ ]:
# Load ground truth
with GT_FILE.open("r", encoding="utf-8") as f:
    gt_rows = json.load(f)

ground_truth_by_uid = {}
gt_mapping_by_uid = {}
gt_perms_by_uid = {}
invalid_gt_uids = []

for row in gt_rows:
    uid = row.get("uid")
    gt_list = row.get("ground_truth_values")
    if uid is None or not isinstance(gt_list, list) or len(gt_list) == 0:
        continue
    normalized = normalize_gt_perms(gt_list)
    if normalized is None:
        invalid_gt_uids.append(uid)
        continue
    value_to_index, gt_perms = normalized
    ground_truth_by_uid[uid] = gt_list
    gt_mapping_by_uid[uid] = value_to_index
    gt_perms_by_uid[uid] = gt_perms

print(f"Loaded ground truth for {len(ground_truth_by_uid):,} UIDs.")
print(f"Invalid GT permutations skipped: {len(invalid_gt_uids):,}")

In [ ]:
# Build task dictionary keyed by uid
# Restrict to temp_1.0 and run_1..run_100

def _is_valid_run_path(path: Path) -> bool:
    parts = path.parts
    if "temp_1.0" not in parts:
        return False
    run_parts = [p for p in parts if p.startswith("run_")]
    if not run_parts:
        return False
    try:
        run_num = int(run_parts[-1].split("_", 1)[1])
    except (ValueError, IndexError):
        return False
    return 1 <= run_num <= 100


tasks = {}
all_result_files = sorted(RESULTS_ROOT.rglob("all_results.json"))
result_files = [p for p in all_result_files if _is_valid_run_path(p)]

invalid_pred_counts = defaultdict(int)
total_pred_counts = defaultdict(int)
invalid_reason_counts = defaultdict(lambda: defaultdict(int))

for result_file in result_files:
    with result_file.open("r", encoding="utf-8") as f:
        rows = json.load(f)

    for row in rows:
        uid = row.get("uid")
        if uid is None or uid not in ground_truth_by_uid:
            continue

        if uid not in tasks:
            tasks[uid] = {
                "uid": uid,
                "id": row.get("id"),
                "category": row.get("category"),
                "subcategory": row.get("subcategory"),
                "prompt_title": row.get("prompt_title"),
                "ground_truth_values": ground_truth_by_uid.get(uid, []),
                "model_perms": defaultdict(list),
            }

        model_name = row.get("model_used")
        extracted = row.get("extracted_value")

        if model_name is None:
            continue

        total_pred_counts[model_name] += 1
        value_to_index = gt_mapping_by_uid.get(uid)
        pred_perm, reason = list_to_perm_indices_checked(extracted, value_to_index)
        if pred_perm is None:
            invalid_pred_counts[model_name] += 1
            invalid_reason_counts[model_name][reason] += 1
            continue

        tasks[uid]["model_perms"][model_name].append(pred_perm)

for uid in tasks:
    tasks[uid]["model_perms"] = dict(tasks[uid]["model_perms"])

print(f"Discovered total result files: {len(all_result_files):,}")
print(f"Using result files (temp_1.0, run_1..run_100): {len(result_files):,}")
print(f"Built task dictionary for {len(tasks):,} UIDs.")

if total_pred_counts:
    invalid_table = (
        pd.DataFrame({
            "model": list(total_pred_counts.keys()),
            "total_predictions": [total_pred_counts[m] for m in total_pred_counts],
            "invalid_predictions": [invalid_pred_counts[m] for m in total_pred_counts],
            "invalid_rate": [
                (invalid_pred_counts[m] / total_pred_counts[m]) if total_pred_counts[m] else 0.0
                for m in total_pred_counts
            ],
        })
        .sort_values(["invalid_rate", "invalid_predictions"], ascending=False)
        .reset_index(drop=True)
    )
    display(invalid_table)

    reason_rows = []
    for model, reasons in invalid_reason_counts.items():
        for reason, count in reasons.items():
            reason_rows.append({
                "model": model,
                "reason": reason,
                "count": count,
            })
    if reason_rows:
        reason_df = (
            pd.DataFrame(reason_rows)
            .sort_values(["model", "count"], ascending=[True, False])
            .reset_index(drop=True)
        )
        display(reason_df)

In [ ]:
# Compute per-uid metrics using Lehmer L0 distributions
records = []
for uid, task in tasks.items():
    gt_perms = gt_perms_by_uid.get(uid)
    if not gt_perms:
        continue

    for model_name, perms in task.get("model_perms", {}).items():
        if len(perms) == 0:
            continue
        rec = {
            "uid": uid,
            "prompt_title": task.get("prompt_title"),
            "model": model_name,
        }
        metrics = perm_distance_metrics(gt_perms, perms)
        if metrics is None:
            continue
        rec.update(metrics)
        records.append(rec)

metrics_df = pd.DataFrame(records)
print(f"Computed per-uid rows: {len(metrics_df):,}")
metrics_df.head()